# Moonshot study on Colab

Reproduces the 2015-2026 pre-registered run: load, assemble, train, and
evaluate against `docs/PREREGISTRATION_2015.md`.

**Quick start:** run section 1, then section 2 **Path A** (skip the Path B
cells), then sections 3–5.

## Read this before choosing a runtime

**Pick a high-CPU runtime, not a GPU one.** That is the opposite of the usual
advice, and it matters, because attaching a T4 makes this notebook *slower*.

| stage | what it is | scales with |
|---|---|---|
| load | read parquet, recompute derived columns | disk, then RAM |
| assemble | pandas merges over ~6M rows | single-core pandas, then RAM |
| train | 30 LightGBM fits (2 models × 5 folds × 3 seeds) | **CPU cores** |
| simulate | Python loop over selected trades | single core |

Nothing in that table wants a GPU.

LightGBM's GPU support accelerates *histogram construction*, which scales with
(features × bins). This panel has **80 features** — small enough that
per-iteration host-to-device transfer dominates, and on narrow data the GPU
build is routinely slower than CPU. It also requires a source build with
`-DUSE_CUDA=1`; `pip install lightgbm` gives you the CPU build whether or not
an accelerator is attached.

Meanwhile the free tier is **2 vCPU**. Feature assembly took 12 minutes on a
4-core machine and would roughly double there.

* **Free tier (2 vCPU):** works, expect ~2× slower than a 4-core box.
* **12-core runtime:** the right choice. Training is 30 independent fits, so it
  is close to linear in cores — roughly 3× faster than 4 cores.
* **GPU runtime:** no benefit to any stage, and you spend quota to get it.

## Memory

The assembled panel is ~1.3 GB at float32, and the merges that build it peak
several times higher. 12 GB is enough. The first attempt at this run was
OOM-killed at float64, which is why the assembler downcasts — LightGBM bins to
255 buckets before choosing a split, so the discarded mantissa cannot change a
tree.

## 1. Install and mount

In [ ]:
# lightgbm and pyarrow are the only extras Colab lacks. The trading calendar
# comes from pandas.tseries.holiday, so no calendar package is needed.
!pip -q install lightgbm pyarrow

import multiprocessing, os, glob, subprocess, sys
print(f"cores: {multiprocessing.cpu_count()}")
!free -g | sed -n 2p

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------- settings
UPLOADED = '/content/drive/MyDrive/Ticker_Data'   # the 8 uploaded files
UA       = 'YOUR NAME your@email.com'             # <-- REQUIRED, real contact
assert '@' in UA, 'the SEC blocks anonymous scrapers, and blocks the whole IP'

# IAI_HOME is the single root; cache/ and store/ derive from it. It must be set
# BEFORE importing iai -- the default is resolved at import time, so setting it
# afterwards silently writes to the ephemeral container instead.
os.environ['IAI_HOME'] = '/content/drive/MyDrive/iai_home'
os.environ['IAI_USER_AGENT'] = UA

# ---------------------------------------------------------------- install
# The repo is private, so `pip install git+https://...` fails with git exit 128
# and no credentials. Install from the source tarball instead -- 232 KB, and it
# keeps a GitHub token out of a notebook that lives on Drive.
SRC = '/content/iai_src'
tarball = f'{UPLOADED}/integratedai_src.tar.gz'
assert os.path.exists(tarball), (
    f'{tarball} not found. Upload integratedai_src.tar.gz into {UPLOADED}.'
)
!rm -rf {SRC} && mkdir -p {SRC} && tar -xzf "{tarball}" -C {SRC}
REPO = glob.glob(f'{SRC}/*/pyproject.toml')[0].rsplit('/', 1)[0]
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', REPO], check=True)

# scripts/ and docs/ are referenced by relative path, so run from the repo root.
os.chdir(REPO)
print('repo:', REPO)

from iai.core.config import Config
_c = Config.moonshot(); _c.ensure_dirs()
print('cache:', _c.data.cache_dir)
print('store:', _c.data.store_dir)
assert 'drive' in str(_c.data.store_dir), \
    'IAI_HOME was set too late. Runtime > Restart session, then run this cell first.'
assert os.path.exists('scripts/moonshot.py') and os.path.exists('docs/PREREGISTRATION_2015.md')
print('ready')

## 2. Get the data — pick ONE path

### Path A (fast): use the prepared files

Put all **seven** files in one Drive folder (`MyDrive/Ticker_Data`) and skip the
fetch entirely:

| file | size | contents |
|---|---|---|
| `w2015_prices_1of5` … `5of5.parquet` | ~19 MiB each | 8,042,901 raw bars, 2015-01-02 → 2025-12-31 |
| `w2015_events.parquet` | 18 MiB | 2,458,729 events (edgar / insiders / flow / news) |
| `members.parquet` | 1.4 MiB | point-in-time universe membership, by quarter |

**115 MB total, down from 610 MB.** Prices ship in five parts because a single
96 MiB file exceeds the transfer limit; the split is by ticker, so each part
holds whole series and the order of the concat does not matter.

Two things were stripped to get there:

* **Derived price columns** (`ret`, `vol`, `adv_usd`, `dollar_vol`,
  `adj_close`, `tradable`) — 293 MB that `add_derived()` recomputes in seconds.
  The cell below calls that same function, so this is not an approximation of
  the pipeline, it *is* the pipeline. Verified across all 8,042,901 rows:
  `tradable` matches exactly, `adj_close` and `adv_usd` to 1e-7, and only 22
  rows differ by more than 1e-6 in `ret` — all of them extreme values (the
  largest a +416,567% move, presumably a reverse split) where the two agree to
  seven significant digits and float32 has no more resolution to give.
* **`payload` and `uid`** — 181 MB that nothing downstream reads. Features come
  from source, kind, ticker, available_ts and weight; `uid` existed only to
  deduplicate at merge time, which has already happened.

The cell asserts the row count and the tradable count, so a missing or
truncated part fails in seconds rather than producing a plausible study on four
fifths of the universe.

### Path B: re-fetch from source

About an hour on one machine, of which the SEC bulk archives are 62 seconds.
Only `prices` and `events` shard usefully — `insiders` downloads 45
whole-market archives, so running it on N machines costs N times the bytes for
none of the speed.

**Run Path A *or* Path B, not both.**

In [ ]:
# ============ PATH A: prepared files ============
# All 7 files live in one Drive folder:
#   w2015_prices_1of5.parquet ... w2015_prices_5of5.parquet
#   w2015_events.parquet
#   members.parquet
UPLOADED = '/content/drive/MyDrive/Ticker_Data'

import glob, os
import pandas as pd
from iai.core.config import Config
from iai.sources.prices import add_derived

cfg = Config.moonshot()
STORE = str(cfg.data.store_dir)

# Locate the parts. Drive folders nest easily -- and a folder inside a
# same-named folder is a very common way to end up one level off -- so if the
# stated path does not hold them, search for them rather than failing on a
# path typo.
parts = sorted(glob.glob(f'{UPLOADED}/w2015_prices_*of5.parquet'))
if len(parts) != 5:
    print(f'{len(parts)} parts at {UPLOADED}; searching MyDrive...')
    found = sorted(glob.glob('/content/drive/MyDrive/**/w2015_prices_*of5.parquet',
                             recursive=True))
    if found:
        UPLOADED = os.path.dirname(found[0])
        parts = sorted(glob.glob(f'{UPLOADED}/w2015_prices_*of5.parquet'))
        print(f'found {len(parts)} parts in {UPLOADED}')
assert len(parts) == 5, (
    f'expected 5 price parts, found {len(parts)}. Upload all five '
    f'w2015_prices_NofN.parquet files, plus w2015_events.parquet and '
    f'members.parquet, into one folder.'
)

px = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
px['ticker'] = px['ticker'].astype(str)
assert len(px) == 8_042_901, f'expected 8,042,901 bars, got {len(px):,} -- a part is truncated'

# Rebuild the derived columns that were stripped to shrink the files. This
# calls the same function the fetcher does, so it is not an approximation of
# the pipeline -- it is the pipeline.
px = add_derived(px, cfg)
px = px.sort_values(['ticker', 'date']).reset_index(drop=True)
px.to_parquet(f'{STORE}/w2015_prices.parquet', index=False)
print(f'prices  {len(px):,} bars, {px.ticker.nunique():,} names, '
      f'{px.date.min().date()} .. {px.date.max().date()}')
assert px['tradable'].sum() == 6_097_117, 'tradable count does not match the source run'
print(f'        tradable {px["tradable"].sum():,}  (matches source run)')
del px

ev = pd.read_parquet(f'{UPLOADED}/w2015_events.parquet')
for c in ['source', 'kind', 'ticker']:
    ev[c] = ev[c].astype(str)
# moonshot.py requests a fixed column list. uid and payload were stripped --
# nothing downstream reads either, and deduplication already happened at merge.
ev['uid'] = range(len(ev))
ev['payload'] = '{}'
ev.to_parquet(f'{STORE}/w2015_events.parquet', index=False)
print(f'events  {len(ev):,}')
print(ev['source'].value_counts().to_string())

MEMBERS = f'{UPLOADED}/members.parquet'
assert os.path.exists(MEMBERS), (
    f'members.parquet not found in {UPLOADED}. It is required -- without it '
    f'every name is treated as enterable on every date, which is lookahead.'
)
print(f'\nready ({UPLOADED}). Skip Path B and go to section 3.')

In [ ]:
SHARD, N_SHARDS = 0, 1   # set per notebook if sharding the price stage
ARGS = f'--start 2015-01-01 --end 2026-01-01 --out {OUT} --user-agent "{UA}"'

!python -m scripts.colab_fetch --stage candidates {ARGS}

In [ ]:
# Slowest stage: ~46 min for 5,490 candidates at 2 req/s on one machine.
# Yahoo answers abuse with blocks rather than 429s, so do not raise --yahoo-rate.
!python -m scripts.colab_fetch --stage prices {ARGS} \
    --shard {SHARD} --n-shards {N_SHARDS} --yahoo-rate 2.0 --workers 6

In [ ]:
# Point-in-time cut to <=2,000 names per quarter, on trailing cap and trailing
# dollar volume. Refuses to run on a partial shard set rather than silently
# screening a different universe.
!python -m scripts.colab_fetch --stage screen {ARGS} --max-names 2000

In [ ]:
!python -m scripts.colab_fetch --stage events   {ARGS} --shard {SHARD} --n-shards {N_SHARDS} --workers 8
!python -m scripts.colab_fetch --stage insiders {ARGS}          # ONE machine only
!python -m scripts.colab_fetch --stage merge    {ARGS} --prefix w2015

!cp {OUT}/w2015_prices.parquet {OUT}/w2015_events.parquet {STORE}/
!ls -la {STORE}

## 3. Sanity checks before spending an hour on training

Cheap, and each one has caught a real bug in this pipeline.

In [ ]:
import pandas as pd

px = pd.read_parquet(f'{STORE}/w2015_prices.parquet', columns=['date','ticker','tradable'])
ev = pd.read_parquet(f'{STORE}/w2015_events.parquet',
                     columns=['source','kind','ticker','event_ts','available_ts'])

print(f"prices  {len(px):,} bars, {px.ticker.nunique():,} names, "
      f"{px.date.min().date()} .. {px.date.max().date()}")
print(f"events  {len(ev):,}\n")
print(ev.source.value_counts().to_string())

# The one that must never fail: nothing may be actionable before it happened.
assert (ev.available_ts >= ev.event_ts).all(), 'LOOKAHEAD: available_ts precedes event_ts'
print('\nPIT check passed: no event is actionable before it occurred')

# Insider events resolved by CIK, not by the filer's typed symbol. Matching on
# the symbol silently dropped 18-30% of open-market buys, zeroed per issuer and
# concentrated on companies that had been renamed.
ins = ev[ev.source == 'insiders']
print(f"insider events {len(ins):,} across {ins.ticker.nunique():,} names")

## 4. Run the pre-registered test

`--members` is not optional. Without it every name in the file is treated as
enterable on every date, including names that only qualified for the universe
years later -- which is lookahead, and the script will warn but still run.

Expect ~15 min of feature assembly then ~30-60 min of training on 4 cores,
less in proportion to how many cores you have.

In [ ]:
import multiprocessing, os

# Path A sets MEMBERS; Path B leaves it to the fetch output directory.
MEMBERS = globals().get('MEMBERS') or f'{OUT}/members.parquet'
assert os.path.exists(MEMBERS), f'members.parquet not found at {MEMBERS}'
LOG = '/content/drive/MyDrive/moonshot_result.log'

# LightGBM takes every core via n_jobs=-1. Training is 30 independent fits
# (2 models x 5 folds x 3 seeds), so this is close to linear in cores -- which
# is why a 12-core runtime is the right choice here and a GPU is not.
print(f'cores available: {multiprocessing.cpu_count()}')

!python scripts/moonshot.py \
    --prefix w2015 \
    --members "{MEMBERS}" \
    --trades-per-week 5 --target 0.10 --stop 0.07 --horizon 10 \
    --max-year-share 0.35 \
    --prereg docs/PREREGISTRATION_2015.md \
    2>&1 | tee "{LOG}"

## 5. Reading the verdict

The criteria are fixed in `docs/PREREGISTRATION_2015.md` and were committed
before this data existed. The script evaluates them itself so the result cannot
be graded after the fact.

**Primary:** net return per trade with a **week-clustered** standard error,
t > 2.0. The clustering matters -- five trades in one week share a regime and
lose together, so dividing by sqrt(n) treats them as five independent draws and
inflates t. The naive t is printed alongside so the size of that dependence is
visible rather than assumed.

**Secondary:** volatility control (skill, not a volatility tilt), outlier
robustness (survives dropping the 20 largest winners), temporal spread (no year
holds >35% of trades), and the EV-vs-P(spike) ablation.

**The stopping rule is part of the pre-registration.** If the primary fails,
the answer is that this edge is not there -- not that the next catalyst class
should be tried. Searching over catalyst classes after a failure is how a
wanted result gets manufactured, and two independent samples already point the
same way.

In [ ]:
print(open(LOG).read()[-3500:])